[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/05_Papers/02_RoPE.ipynb)

# RoPE: Rotary Position Embeddings

This notebook implements **RoPE** (Su et al., 2021), the position encoding used in virtually
every modern LLM — LLaMA, Mistral, GPT-NeoX, Qwen, Gemma, DeepSeek. RoPE encodes position
by **rotating** query and key vectors, so that attention scores naturally depend on relative
distance between tokens.

We will:
1. Understand why attention needs position information
2. Compare absolute, learned, and relative position encodings
3. Derive the RoPE rotation math
4. Implement RoPE from scratch (both explicit and efficient versions)
5. Verify the relative position property
6. Visualize attention patterns with RoPE

**Reference:** Su, J. et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding.*
https://arxiv.org/abs/2104.09864

In [ ]:
import sys, os

# In Colab, clone the repo so local imports (src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/05_Papers")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.utils.device import set_seed

set_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 1. The Position Problem

Self-attention is **permutation-invariant**: $\text{softmax}(QK^T/\sqrt{d})V$ produces
the same output regardless of token order. Without position information:

- "The **cat** sat on the mat" and "mat the on sat **cat** The" produce identical attention patterns
- The model has no way to distinguish "A before B" from "B before A"

**Three approaches to inject position:**

| Method | How | Used by | Limitation |
|---|---|---|---|
| Sinusoidal (absolute) | Add fixed sin/cos to embeddings | Original Transformer | Doesn't generalize to longer sequences |
| Learned (absolute) | Add learned position embeddings | BERT, GPT-2 | Fixed max length, no extrapolation |
| **Rotary (relative)** | **Rotate Q, K vectors** | **LLaMA, Mistral, all modern LLMs** | **Generalizes, captures relative distance** |

## 2. Absolute vs Relative Position

**Absolute encoding** (BERT, GPT-2): $\tilde{x}_m = x_m + p_m$ where $p_m$ is a fixed or
learned vector for position $m$. The attention score becomes:

$q_m^T k_n = (x_m + p_m)^T W_q^T W_k (x_n + p_n)$

This contains absolute positions $p_m$ and $p_n$ separately, not just the distance $m - n$.

**Relative encoding** (RoPE): We want the attention score to depend on the **relative
distance** $m - n$, not the absolute positions. RoPE achieves this by rotating Q and K:

$\langle R_m q, R_n k \rangle = \langle R_{m-n} q, k \rangle$

where $R_m$ is a rotation matrix. The dot product only depends on $m - n$.

In [ ]:
# Demonstrate: vanilla attention is permutation-invariant
d = 8
W_q = torch.randn(d, d)
W_k = torch.randn(d, d)

# Two sequences with tokens in different order
x = torch.randn(4, d)  # 4 tokens
x_shuffled = x[[2, 0, 3, 1]]  # shuffled order

# Attention scores
Q, K = x @ W_q, x @ W_k
Q_s, K_s = x_shuffled @ W_q, x_shuffled @ W_k

attn = (Q @ K.T) / math.sqrt(d)
attn_s = (Q_s @ K_s.T) / math.sqrt(d)

# The scores between the SAME token pairs are identical
# Token 0 attending to Token 1: attn[0,1] should equal attn_s[1,3] (where they moved to)
print("Attention score between tokens 0 and 1:")
print(f"  Original order:  attn[0,1] = {attn[0,1].item():.4f}")
print(f"  Shuffled order:  attn[1,3] = {attn_s[1,3].item():.4f}")
print(f"  Same? {torch.allclose(attn[0,1], attn_s[1,3])}")
print("\n→ Attention doesn't know position. It gives the same score regardless of where tokens are.")

## 3. RoPE's Key Idea: Rotation Encodes Position

In 2D, a rotation by angle $\theta$ is:

$R_\theta = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$

RoPE applies rotation to each **pair of dimensions** in the query and key vectors.
For position $m$, dimensions $(2i, 2i+1)$ are rotated by angle $m \cdot \theta_i$:

$\theta_i = 10000^{-2i/d}$

This is the same frequency formula as sinusoidal encodings! But instead of **adding**
sin/cos to the embedding, RoPE **rotates** the Q/K vectors.

**Why rotation?** Because rotations preserve dot products in a useful way:

$\langle R_m q, R_n k \rangle = \langle R_{m-n} q, k \rangle$

The attention score between positions $m$ and $n$ depends only on the relative
distance $m - n$, not the absolute positions.

## 4. The Rotation Matrix

For a $d$-dimensional vector, RoPE applies $d/2$ independent 2D rotations.
Each pair of dimensions $(2i, 2i+1)$ gets its own rotation frequency:

$\theta_i = 10000^{-2i/d}$

At position $m$, the full rotation is:

$R_m = \text{diag}\begin{pmatrix} R_{m\theta_0} & R_{m\theta_1} & \cdots & R_{m\theta_{d/2-1}} \end{pmatrix}$

where each $R_{m\theta_i}$ is a 2x2 rotation matrix applied to dimensions $(2i, 2i+1)$.

**Low-frequency dimensions** (small $i$) rotate slowly → capture **long-range** position.
**High-frequency dimensions** (large $i$) rotate fast → capture **local** position.

In [ ]:
# Visualize: 2D rotation at different positions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: rotate a single vector at different positions
ax = axes[0]
v = np.array([1.0, 0.0])  # unit vector
theta = 0.3  # rotation frequency

colors = plt.cm.viridis(np.linspace(0, 0.9, 8))
for pos in range(8):
    angle = pos * theta
    rotated = np.array([v[0]*np.cos(angle) - v[1]*np.sin(angle),
                        v[0]*np.sin(angle) + v[1]*np.cos(angle)])
    ax.arrow(0, 0, rotated[0]*0.9, rotated[1]*0.9,
             head_width=0.05, head_length=0.03, color=colors[pos])
    ax.annotate(f'pos={pos}', xy=(rotated[0]*1.05, rotated[1]*1.05),
                fontsize=9, color=colors[pos])

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('2D Rotation at Different Positions', fontsize=14)
ax.set_xlabel('Dimension 0', fontsize=12)
ax.set_ylabel('Dimension 1', fontsize=12)

# Right: frequency bands
ax = axes[1]
d = 64
positions = np.arange(50)
for i, label in [(0, 'dim 0-1 (slow)'), (8, 'dim 16-17'), (16, 'dim 32-33'), (31, 'dim 62-63 (fast)')]:
    freq = 10000 ** (-2 * i / d)
    angles = positions * freq
    ax.plot(positions, np.cos(angles), label=label, linewidth=2)

ax.set_xlabel('Position', fontsize=12)
ax.set_ylabel('cos(m · θ_i)', fontsize=12)
ax.set_title('Frequency Bands (d=64)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Why Rotations Work

The key property: for rotation matrices $R_m$ and $R_n$:

$\langle R_m q, R_n k \rangle = \langle R_{m-n} q, k \rangle$

**Proof (2D case):**

$(R_m q)^T (R_n k) = q^T R_m^T R_n k = q^T R_{-m} R_n k = q^T R_{n-m} k$

Since $R_\theta^T = R_{-\theta}$ (rotation matrices are orthogonal) and $R_a R_b = R_{a+b}$.

This means the attention score between positions $m$ and $n$ depends only on $m - n$.
Position 5 attending to position 3 gives the **same score** as position 10 attending
to position 8 — both have relative distance 2.

## 6. Efficient Implementation

Building the full block-diagonal rotation matrix is wasteful. Instead, we:
1. Split Q/K into pairs of dimensions: $(q_0, q_1), (q_2, q_3), ...$
2. Precompute $\cos(m \cdot \theta_i)$ and $\sin(m \cdot \theta_i)$ for all positions and frequencies
3. Apply the rotation element-wise:

```
q_rotated[..., 0::2] = q[..., 0::2] * cos - q[..., 1::2] * sin
q_rotated[..., 1::2] = q[..., 0::2] * sin + q[..., 1::2] * cos
```

This is how LLaMA and all modern LLMs implement it — no matrix multiplication needed.

In [ ]:
def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    """
    Precompute the cos/sin frequency table for RoPE.
    
    Args:
        dim: Head dimension (must be even)
        max_seq_len: Maximum sequence length
        base: Base for the frequency computation (default 10000)
    
    Returns:
        cos, sin: tensors of shape [max_seq_len, dim//2]
    """
    # θ_i = base^(-2i/d) for i = 0, 1, ..., d/2 - 1
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    
    # Outer product: positions × frequencies → angles
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, freqs)  # [seq_len, dim//2]
    
    return angles.cos(), angles.sin()


# Demo
cos_table, sin_table = precompute_rope_freqs(dim=8, max_seq_len=16)
print(f"cos_table shape: {cos_table.shape}  (seq_len × dim//2)")
print(f"sin_table shape: {sin_table.shape}")
print(f"\nFrequencies for dim=8: {1.0 / (10000 ** (torch.arange(0, 8, 2).float() / 8))}")
print(f"  dim 0-1: freq = 1.000 (fast rotation → local position)")
print(f"  dim 6-7: freq = 0.018 (slow rotation → long-range position)")

In [ ]:
def apply_rope(q, k, cos, sin):
    """
    Apply Rotary Position Embeddings to Q and K tensors.
    
    Args:
        q, k: tensors of shape [..., seq_len, dim]
        cos, sin: precomputed tables of shape [seq_len, dim//2]
    
    Returns:
        q_rotated, k_rotated: same shape as input
    """
    seq_len = q.shape[-2]
    cos = cos[:seq_len]  # [seq_len, dim//2]
    sin = sin[:seq_len]
    
    def rotate(x):
        # Split into even/odd dimension pairs
        x_even = x[..., 0::2]  # dims 0, 2, 4, ...
        x_odd = x[..., 1::2]   # dims 1, 3, 5, ...
        
        # Apply 2D rotation to each pair
        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos
        
        # Interleave back
        rotated = torch.stack([rotated_even, rotated_odd], dim=-1)
        return rotated.flatten(-2)  # [..., seq_len, dim]
    
    return rotate(q), rotate(k)


# Demo
d = 8
seq_len = 6
cos_tab, sin_tab = precompute_rope_freqs(dim=d, max_seq_len=32)

q = torch.randn(1, seq_len, d)  # [batch, seq, dim]
k = torch.randn(1, seq_len, d)

q_rope, k_rope = apply_rope(q, k, cos_tab, sin_tab)
print(f"Input Q shape:  {q.shape}")
print(f"Output Q shape: {q_rope.shape}")
print(f"\nQ values changed (rotation applied): {not torch.allclose(q, q_rope)}")

In [ ]:
# VERIFY: attention score depends only on relative distance
d = 16
cos_tab, sin_tab = precompute_rope_freqs(dim=d, max_seq_len=64)

# Create the SAME q and k vectors but at DIFFERENT absolute positions
q_vec = torch.randn(d)  # a fixed query vector
k_vec = torch.randn(d)  # a fixed key vector

# Test: score(pos_m, pos_n) should equal score(pos_m+5, pos_n+5) if m-n is the same
results = []
for offset in [0, 5, 10, 20]:
    m, n = 3 + offset, 1 + offset  # relative distance always = 2
    
    # Place q at position m, k at position n
    q_at_m = q_vec.unsqueeze(0).unsqueeze(0)  # [1, 1, d]
    k_at_n = k_vec.unsqueeze(0).unsqueeze(0)
    
    # Manually apply rotation for specific positions
    cos_m = cos_tab[m:m+1]  # [1, d//2]
    sin_m = sin_tab[m:m+1]
    cos_n = cos_tab[n:n+1]
    sin_n = sin_tab[n:n+1]
    
    q_rot, _ = apply_rope(q_at_m, q_at_m, cos_tab[m:m+1], sin_tab[m:m+1])
    _, k_rot = apply_rope(k_at_n, k_at_n, cos_tab[n:n+1], sin_tab[n:n+1])
    
    score = (q_rot @ k_rot.transpose(-2, -1)).item() / math.sqrt(d)
    results.append((m, n, m-n, score))

print("Attention scores with SAME relative distance (m-n=2) at different absolute positions:")
print(f"{'pos_m':>6s} {'pos_n':>6s} {'m-n':>5s} {'score':>10s}")
for m, n, rel, score in results:
    print(f"{m:6d} {n:6d} {rel:5d} {score:10.6f}")

scores = [r[3] for r in results]
print(f"\nAll scores equal? {all(abs(s - scores[0]) < 1e-5 for s in scores)}")
print("→ RoPE makes attention depend only on relative distance, not absolute position.")

In [ ]:
# Compare: no position vs absolute position vs RoPE
d = 16
seq_len = 8
cos_tab, sin_tab = precompute_rope_freqs(dim=d, max_seq_len=32)

torch.manual_seed(42)
x = torch.randn(1, seq_len, d)
W_q = nn.Linear(d, d, bias=False)
W_k = nn.Linear(d, d, bias=False)

Q = W_q(x)  # [1, seq, d]
K = W_k(x)

# 1. No position encoding
attn_none = (Q @ K.transpose(-2, -1)) / math.sqrt(d)

# 2. Sinusoidal (absolute) position encoding — add to input
pos_enc = torch.zeros(seq_len, d)
for pos in range(seq_len):
    for i in range(0, d, 2):
        freq = 10000 ** (-i / d)
        pos_enc[pos, i] = math.sin(pos * freq)
        pos_enc[pos, i+1] = math.cos(pos * freq)
Q_abs = W_q(x + pos_enc.unsqueeze(0))
K_abs = W_k(x + pos_enc.unsqueeze(0))
attn_abs = (Q_abs @ K_abs.transpose(-2, -1)) / math.sqrt(d)

# 3. RoPE — rotate Q, K
Q_rope, K_rope = apply_rope(Q, K, cos_tab, sin_tab)
attn_rope = (Q_rope @ K_rope.transpose(-2, -1)) / math.sqrt(d)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, attn, title in zip(axes,
    [attn_none[0], attn_abs[0], attn_rope[0]],
    ['No Position', 'Sinusoidal (Absolute)', 'RoPE (Relative)']):
    im = ax.imshow(attn.detach().numpy(), cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
plt.colorbar(im, ax=axes, shrink=0.8)
plt.suptitle('Attention Scores: No Position vs Absolute vs RoPE', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. RoPE in Modern LLMs

Every major open-source LLM uses RoPE:

| Model | Base Freq | Head Dim | Context Length |
|---|---|---|---|
| LLaMA 1 | 10,000 | 128 | 2,048 |
| LLaMA 2 | 10,000 | 128 | 4,096 |
| LLaMA 3 | 500,000 | 128 | 128,000 |
| Mistral 7B | 10,000 | 128 | 32,768 |
| Qwen 2 | 1,000,000 | 128 | 131,072 |
| DeepSeek-V3 | 10,000 | 128 | 128,000 |

Note: LLaMA 3 and Qwen use a much higher base frequency — this stretches the rotation
periods so positions further apart still get distinct angles. This is one way to support
longer context windows.

## 8. Length Extrapolation

RoPE naturally supports **longer sequences than seen during training** via frequency scaling:

- **Linear scaling** (simplest): divide all frequencies by the scaling factor. If trained on 4K, scale by 4x to support 16K.
- **NTK-aware scaling**: increase the base frequency instead of dividing all frequencies equally. Preserves high-frequency resolution.
- **YaRN** (Yet another RoPE extensioN): combines NTK scaling with an attention temperature adjustment.
- **Dynamic NTK**: adaptively changes the base frequency based on sequence length at inference time.

This is why modern LLMs can handle very long contexts — the RoPE frequencies are
designed to extrapolate smoothly.

In [ ]:
# Demo: full attention with RoPE on a small sequence
d = 32
n_heads = 4
head_dim = d // n_heads
seq_len = 12

cos_tab, sin_tab = precompute_rope_freqs(dim=head_dim, max_seq_len=64)

torch.manual_seed(0)
x = torch.randn(1, seq_len, d)

# Project to Q, K, V
W_q = nn.Linear(d, d, bias=False)
W_k = nn.Linear(d, d, bias=False)
W_v = nn.Linear(d, d, bias=False)

Q = W_q(x).view(1, seq_len, n_heads, head_dim).transpose(1, 2)  # [1, heads, seq, head_dim]
K = W_k(x).view(1, seq_len, n_heads, head_dim).transpose(1, 2)
V = W_v(x).view(1, seq_len, n_heads, head_dim).transpose(1, 2)

# Apply RoPE to each head
Q_rope, K_rope = apply_rope(Q, K, cos_tab, sin_tab)

# Compute attention
attn_weights = torch.softmax(Q_rope @ K_rope.transpose(-2, -1) / math.sqrt(head_dim), dim=-1)
output = attn_weights @ V

print(f"Multi-head attention with RoPE:")
print(f"  Input:   {x.shape}")
print(f"  Q/K/V:   {Q.shape} (per head)")
print(f"  Attn:    {attn_weights.shape}")
print(f"  Output:  {output.shape}")

In [ ]:
# Visualize attention pattern with RoPE — should show locality bias
fig, axes = plt.subplots(1, n_heads, figsize=(16, 4))

for h in range(n_heads):
    ax = axes[h]
    im = ax.imshow(attn_weights[0, h].detach().numpy(), cmap='viridis', vmin=0)
    ax.set_title(f'Head {h}', fontsize=12)
    ax.set_xlabel('Key position')
    if h == 0:
        ax.set_ylabel('Query position')

plt.colorbar(im, ax=axes, shrink=0.8)
plt.suptitle('Attention Patterns with RoPE (4 heads, seq_len=12)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Note: Some heads may show locality bias (attending to nearby positions),")
print("while others attend more broadly. This is expected — different heads learn")
print("different position-dependent patterns.")

## 9. Key Takeaways

1. **RoPE encodes position by rotating Q and K vectors.** Each pair of dimensions is rotated by an angle proportional to the position, with different frequencies for different dimension pairs.

2. **Attention scores depend on relative distance, not absolute position.** The rotation property $\langle R_m q, R_n k \rangle = \langle R_{m-n} q, k \rangle$ ensures this mathematically.

3. **Low-frequency dimensions capture long-range position, high-frequency dimensions capture local position.** The frequency formula $\theta_i = 10000^{-2i/d}$ creates a geometric progression from fast to slow rotation.

4. **RoPE generalizes to longer sequences** via frequency scaling (linear, NTK, YaRN). This is how LLaMA 3 handles 128K context despite being trained on shorter sequences.

5. **RoPE is applied before the attention computation.** It modifies Q and K before the dot product. This connects to the Flash Attention notebook — RoPE is computed first, then the tiled attention kernel runs on the rotated Q/K.

### Further Reading

- Su et al. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding.* https://arxiv.org/abs/2104.09864
- blog.eleuther.ai: *Rotary Embeddings: A Relative Revolution* https://blog.eleuther.ai/rotary-embeddings/
- Peng et al. (2023). *YaRN: Efficient Context Window Extension.* https://arxiv.org/abs/2309.00071